Tcp world vs tcp tool

How to align the gripper?

Principle, set good defaults!

At default [0,0,0,0,0,0] I want a resonabel position

Lets use TCP tool as that aligns with the camera frame, z pointing into the objects

Lets align the width of the gripper with the y axis of camera? so at 0 z deg its flat like the x axis...

In [ ]:
import numpy as np
import open3d as o3d
from transforms3d.euler import euler2mat

from graspnetAPI import GraspNetEval, Grasp, GraspGroup
import graspnetAPI

%reload_ext autoreload
%autoreload 2

In [ ]:
from bam_utils.transforms import xyzrpy_to_matrix

In [ ]:
def make_graspnet_grasp_V2(action) -> graspnetAPI.Grasp:
    finger_width = 0.02
    finger_height = 0.02
    # assert finger_height >= 0.02 # or will have rending problems

    x, y, z, rx, ry, rz, grasp_width = action

    # Internally graspnet uses a strange coordinate system, see: bam_gym/docs/grasp_net_def.png

    height = finger_width
    depth = finger_height - 0.02# minus 2cm to account for depth_base, it will be added back later internally in GraspNet

    T_world_tcp = xyzrpy_to_matrix([x, y, z], [rx, ry, rz])  # Convert to transformation matrix
    grasp_dummy = graspnetAPI.Grasp()
    grasp_dummy.depth = depth
    # T_world_graspnet = T_world_tcp @ graspnetAPI.Grasp().T_tcp_graspnet  #[BUG] you cannot use a blank grasp like this..
    T_world_graspnet = T_world_tcp @ grasp_dummy.T_tcp_graspnet  
    R_world_graspnet = T_world_graspnet[:3, :3] 
    t_world_graspnet = T_world_graspnet[:3, 3]

    # BUG ok the issue is the the translation of t_world is not big enough.. it needs to be shifted by 2 as well

    score = 0
    object_id = -1

    # [score, width, height, depth, rotation_matrix(9), translation(3), object_id]
    grasp_params = [score, grasp_width, height, depth, R_world_graspnet, t_world_graspnet, object_id]
    print(grasp_params)
    return graspnetAPI.Grasp(*grasp_params)

print(graspnetAPI.Grasp().T_graspnet_tcp)

In [ ]:
grasp = make_graspnet_grasp_V2([0, 0, 0, 0, 0, 0, 0.05])
assert np.allclose(grasp.T_graspnet_tcp @ grasp.T_tcp_graspnet, np.eye(4))
print(grasp.T_graspnet_tcp)
graspnet_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(0.01)
graspnet_frame.transform(grasp.transform_matrix)
tcp_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(0.01)
tcp_frame.transform(grasp.tcp_frame)
origin_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(0.02)


origin_frame.paint_uniform_color([0, 0, 0])    # Blue

o3d.visualization.draw_geometries([*grasp.to_open3d_geometry(), graspnet_frame, origin_frame, tcp_frame])

In [ ]:
def make_graspnet_grasp_V1(x = 0., y = 0., z = 0., rx = 0., ry = 0., rz = 0., grasp_width = 0.1, finger_width = 0.02, finger_height = 0.04, score = 0.0):
    """ 
    Convert from BAM grasp convention to GraspNet grasp convention.
    
    Graspnet. See the label image

    - z axis goes out the side
    - center is 2cm from base (depth_base = 0.02) and depth from finger tip

    By default they use different total lengths of fingers (depth_base + depth), which is interest, I see 3cm long and 5cm long fingers
    """

    height = finger_width
    depth = finger_height - 0.02 # minus 2cm to account for depth_base, it will be added back later internally in GraspNet

    rotation_matrix = euler2mat(rx, ry, rz, axes='sxyz')  # rotation matrix
    # T_tcp_tool = euler2mat(0, -np.pi/2, 0, axes='sxyz')  # Aligns correct along z
    T_tcp_tool = euler2mat(0, -np.pi/2, np.pi/2, axes='sxyz')  # Aligns correct along z and x
    rotation_matrix = T_tcp_tool @ rotation_matrix  

    translation = np.array([x, y, z])
    translation += np.array([0, 0, -depth])  # Adjust translation to the center of the grasp

    object_id = -1

    # [score, width, height, depth, rotation_matrix(9), translation(3), object_id]
    grasp_params = [score, grasp_width, height, depth, rotation_matrix, translation, object_id]
    return Grasp(*grasp_params)

In [ ]:
g_default = Grasp()
g_param_default = grasp_net_grasp()

In [ ]:
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(0.1)
o3d.visualization.draw_geometries([g_default.to_open3d_geometry(), frame])


In [ ]:
o3d.visualization.draw_geometries([g_param_default.to_open3d_geometry(), frame])

In [ ]:
g = grasp_net_grasp(grasp_width=0.05, finger_width=0.02)
o3d.visualization.draw_geometries([g.to_open3d_geometry(use_defaults=False), frame])

In [ ]:
g = grasp_net_grasp(finger_height=0)
o3d.visualization.draw_geometries([g.to_open3d_geometry(use_defaults=False), frame])